# A3.2 · Sandboxed execution

**Function A — Securing AI Architectures → Securing the Architecture — Runtime and the Gateway**  ·  *Security of AI*

Builds on **[A3.1 · Default-deny on the tool call](https://spbreed.github.io/cyber-commons/lessons/A3.1.html)**.

| | |
|---|---|
| Tools used | Kubernetes NetworkPolicy, seccomp, Terraform, gVisor |

## What this lesson is

**What it covers.** Run the same code inside and outside the sandbox and enumerate what each could reach.

**Why a security engineer needs it.** Model-authored code inherits the runtime's reach, including any credential mounted into the environment. The control it builds is: execution in an isolate with no ambient credentials, a bounded filesystem and no default network.

This is a **control** lesson: it builds the mechanism, then breaks it, so you can see what the control is actually load-bearing for rather than taking the claim on trust.

## 1 · The hook

"It runs in a sandbox" is not a control until a manifest says what the sandbox contains. A container with the host network and a mounted socket is a deployment convenience wearing the word — and a namespace with no default-deny NetworkPolicy is default-allow, however many narrow policies you wrote.

> **At CyberTravels.** The Coding Agent runs generated code, and the File System Agent runs on Alex's laptop. “It runs in a sandbox” is not a control until somebody says whether that sandbox can see `~/.aws` and the HR folder. R6.

## 2 · The framework

```
   "it runs in a sandbox"  ->  contains what, exactly?

   filesystem   only the workspace, or the host's?
   network      none, allowlist, or the host's network namespace?
   sockets      is the container runtime socket mounted in?
   syscalls     full kernel surface, or a filtered one?
   credentials  is a production token mounted inside it?

   and the network answer is two Kubernetes objects, not one:

     NetworkPolicy default-deny-all   podSelector: {}     <- makes
       policyTypes: [Ingress, Egress]                        every pod
                                                             RESTRICTED
     NetworkPolicy workflow-agent-egress                   <- adds back
       egress: bookings-db:5432, kube-dns:53                  two hops

   policies are additive allow-lists. delete the first and the
   second stops being a restriction, silently, with nothing failing
```

**Mitigates: T11 Malicious Code Execution · T2 Tool Misuse.**

For an agent that runs code, the sandbox **is** the security boundary. Not the
prompt, not the code review, not the model's training. The question is never
"is there a sandbox" but "what does this one actually contain".

A1.8's lesson was that reach is a property of the environment, not of intent —
the benign task touched a private key because `open()` sees what the process
sees. So the control is to change what the process sees.

Four dimensions, and the fourth is the one teams get wrong:

**Filesystem.** A bounded working directory. Not the home directory, which holds
`.ssh`, `.aws` and `.config`.

**Process and syscall.** No spawning, no ptrace, resource ceilings so a runaway
loop is contained rather than fatal.

**Network.** No egress by default. Not "restricted" — none, and then an
explicit allow per destination the workload genuinely needs.

**Credentials.** The one people miss: **a sandbox with production credentials
mounted in it is not a sandbox.** Isolation of the filesystem is irrelevant if
the environment holds a token that reaches production over a network the
sandbox does permit. The strongest boundary in the world does not help when the
keys are inside it.

### Say it in the manifest, or you have not said it

"No egress by default" is a claim about a cluster, and a claim about a cluster
is worth what its manifest says. In Kubernetes that is two objects and they are
both required, because a `NetworkPolicy` is an **additive allow-list**: pods
that no policy selects are unrestricted, and policies never deny — they only
add permitted traffic to a pod that some policy has already made restricted.

So the pattern is: one policy that selects every pod in the namespace and
permits nothing, then one narrow policy per destination the agent needs. Delete
the first and the second stops being a restriction at all, silently, with every
pod still running and every test still green.

The same two-object shape appears in every cloud. On AWS a security group with
no egress rules plus one rule per endpoint, and an IAM policy whose `Condition`
binds the role to the workload identity rather than to a subnet. On GCP a
hierarchical firewall policy with a low-priority `deny` on `0.0.0.0/0` and
higher-priority `allow` rules. Different nouns, identical structure: deny
everything by construction, then name what is permitted, one destination at a
time.

> **What this control closes.**
>
> Changes what the executing process can **reach**, which is the only variable A1.8 turned on. A sandbox holding production credentials contains nothing that matters.

## 3 · The network dimension, as a manifest

Two objects, both required. The first makes every pod in the namespace
restricted and permits nothing; the second adds back exactly one destination.

```yaml
# 1. Default-deny. Selects EVERY pod in the namespace, permits no egress and
#    no ingress. Without this object the policy below is not a restriction —
#    it is an allowance on a pod that was already unrestricted.
apiVersion: networking.k8s.io/v1
kind: NetworkPolicy
metadata:
  name: default-deny-all
  namespace: prod-agents
spec:
  podSelector: {}                 # every pod
  policyTypes: [Ingress, Egress]  # both, or egress stays wide open
---
# 2. One destination, for one agent, selected by the same service account that
#    carries its SPIFFE ID. DNS is separate and explicit: without port 53 the
#    agent cannot resolve anything, which is a correct default and a confusing
#    first afternoon.
apiVersion: networking.k8s.io/v1
kind: NetworkPolicy
metadata:
  name: workflow-agent-egress
  namespace: prod-agents
spec:
  podSelector:
    matchLabels:
      app.kubernetes.io/name: workflow-agent   # sa/workflow-agent
  policyTypes: [Egress]
  egress:
    - to:
        - namespaceSelector:
            matchLabels: {kubernetes.io/metadata.name: prod-data}
          podSelector:
            matchLabels: {app: bookings-db}
      ports:
        - {protocol: TCP, port: 5432}
    - to:
        - namespaceSelector:
            matchLabels: {kubernetes.io/metadata.name: kube-system}
          podSelector:
            matchLabels: {k8s-app: kube-dns}
      ports:
        - {protocol: UDP, port: 53}
```

Note what is **not** in the allow-list, and what that costs an attacker:
`169.254.169.254` — the cloud metadata service, which hands out the node's
credentials to anything that can reach it — is unreachable because nothing
named it, not because anybody thought of it. That is the property a deny-list
can never have.

The pod itself carries the other three dimensions:

```yaml
spec:
  serviceAccountName: workflow-agent
  automountServiceAccountToken: false   # no ambient cluster credential
  securityContext:
    runAsNonRoot: true
    runAsUser: 10001
    seccompProfile: {type: RuntimeDefault}
  containers:
    - name: agent
      securityContext:
        allowPrivilegeEscalation: false
        readOnlyRootFilesystem: true
        capabilities: {drop: [ALL]}
      resources:
        limits: {cpu: "1", memory: 1Gi}
      volumeMounts:
        - {name: work, mountPath: /sandbox/work}   # the only writable path
  volumes:
    - name: work
      emptyDir: {sizeLimit: 512Mi}
```

The same shape on **AWS** — a security group whose egress rules are the whole
allow-list, and a role whose trust policy binds to the workload rather than to
the subnet:

```hcl
resource "aws_security_group" "workflow_agent" {
  name   = "workflow-agent"
  vpc_id = var.vpc_id
  # No `egress` block at all: an AWS security group with no egress rules
  # permits nothing outbound. The default SG that ships with a VPC allows
  # 0.0.0.0/0 — never attach that one.
}

resource "aws_vpc_security_group_egress_rule" "bookings_db" {
  security_group_id            = aws_security_group.workflow_agent.id
  referenced_security_group_id = aws_security_group.bookings_db.id
  ip_protocol                  = "tcp"
  from_port                    = 5432
  to_port                      = 5432
}
```

```json
{
  "Version": "2012-10-17",
  "Statement": [{
    "Effect": "Allow",
    "Action": ["s3:GetObject"],
    "Resource": "arn:aws:s3:::cybertravels-itineraries/*",
    "Condition": {
      "StringEquals": {
        "aws:PrincipalTag/spiffe-id":
          "spiffe://cybertravels.com/ns/prod/sa/workflow-agent"
      },
      "Bool": {"aws:SecureTransport": "true"}
    }
  }]
}
```

And on **GCP**, where the deny is explicit and priority-ordered rather than
implicit:

```yaml
# gcloud compute network-firewall-policies rules create ...
- priority: 65000            # lowest priority: the floor
  direction: EGRESS
  action: deny
  match: {destIpRanges: ["0.0.0.0/0"]}
- priority: 1000             # higher priority wins
  direction: EGRESS
  action: allow
  targetSecureTags: ["tagValues/workflow-agent"]
  match:
    destIpRanges: ["10.20.0.0/24"]
    layer4Configs: [{ipProtocol: tcp, ports: ["5432"]}]
```

Three products, one structure: deny everything by construction, then name what
is permitted, one destination at a time.

## 4 · The check, as a skill

One probe, three environments, and the middle one is the configuration that actually ships: real isolation with production credentials mounted inside it. The skill then evaluates the NetworkPolicy objects connection by connection, because a namespace no policy selects is unrestricted rather than unconfigured.

In [ ]:
# skills/runtime/sandbox-containment-probe/SKILL.md — embedded verbatim from the repository.
# This is the file itself, not a paraphrase of it.
SKILL_MD = r"""---
name: sandbox-containment-probe
description: >-
  Execute the same code against an unsandboxed environment, a sandbox with
  production credentials mounted, and a contained one — then evaluate the
  network policies that separate them, connection by connection. Use when
  choosing or reviewing a runtime for agent-authored code.
allowed-tools: Read, Grep, Glob, Bash
---

# A sandbox with production credentials in it is a directory

"Sandboxed" is not a property a runtime has; it is a property of a specific
configuration, and the configuration that defeats it is common: the isolation
is real and the credentials were mounted in anyway. Probing all three
environments with one piece of code is what makes the difference visible.

## When to use this

Before choosing a runtime for model-authored code, and after any change to what
is mounted into it.

## Procedure

**1 — Write one probe and hold it fixed.** It should read the environment, walk
the filesystem, and attempt a connection. Varying the probe per environment
tests probes; varying the environment tests containment.

**2 — Run it unsandboxed and record the reach.** This is the baseline the
sandbox is being asked to reduce. It usually includes key material nobody
remembered was on that host.

**3 — Run it in the sandbox as actually configured.** Not the reference
configuration — the one in your manifest, with whatever is mounted. Credentials
and a route to production are the two things to look for.

**4 — Run it in the contained configuration** and record what is left. That
delta is the value of the control, and it is the number to put in the change
request.

**5 — Evaluate the network policy connection by connection.** For each
(source, destination, port) the workload might attempt, does a policy permit
it? Kubernetes NetworkPolicy is default-allow until a policy selects the pod,
so a namespace with no policy is not "no rules" — it is "all traffic".

## Output contract

```json
{
  "probe": "str",
  "environments": [{"name": "unsandboxed|sandboxed|contained",
                    "filesystem": ["str"], "credentials": ["str"], "network": ["str"]}],
  "delta": {"removed": ["str"], "remaining": ["str"]},
  "network_policy": {"attempts": [{"from": "str", "to": "str", "port": 0, "permitted": false}],
                     "default_allow_namespaces": ["str"]}
}
```

## Failure modes

- **Testing the reference configuration.** Test the one in the manifest.
- **Calling isolation containment while credentials are mounted.** The process
  boundary did not stop being crossed by a file.
- **Reading a namespace with no NetworkPolicy as restricted.** It is
  unrestricted.
"""

In [ ]:
# Execute the skill above, using the shared runtime rather than a copy.
import glob, importlib.util, os, sys

# Kaggle mounts an attached kernel under /kaggle/input, and it uses two
# different layouts — /kaggle/input/<slug>/ on some kernels and
# /kaggle/input/notebooks/<user>/<slug>/ on others. Both were observed on the
# same account in the same hour, so match either. The recursive glob is cheap
# here because /kaggle/input holds only what is attached; globbing the working
# tree instead cost eleven seconds a notebook.
_WHERE = (sorted(glob.glob("/kaggle/input/**/cyber-commons-skill-runtime/__script__.py",
                           recursive=True))
          + [os.path.join(p, "skills/_runtime/cyber_commons_skill_runtime.py")
             for p in (".", "..", "../..")])
_found = next((p for p in _WHERE if os.path.isfile(p)), None)
if _found is None:
    # Say what was looked for and what is actually there. "The runtime is
    # missing" on its own costs whoever hits it an afternoon.
    raise SystemExit("The shared skill runtime is missing."
                     "  looked at: " + repr(_WHERE) +
                     "  /kaggle/input holds: " +
                     repr(glob.glob("/kaggle/input/**", recursive=True)[:20]) +
                     "  cwd: " + os.getcwd() +
                     ". On Kaggle it is attached to this notebook as a "
                     "source; locally it is skills/_runtime/ in the repository.")
_spec = importlib.util.spec_from_file_location("cyber_commons_skill_runtime", _found)
cyber_commons_skill_runtime = importlib.util.module_from_spec(_spec)
sys.modules["cyber_commons_skill_runtime"] = cyber_commons_skill_runtime
_spec.loader.exec_module(cyber_commons_skill_runtime)
from cyber_commons_skill_runtime import run_skill

# Split skills/runtime/sandbox-containment-probe/SKILL.md into the two halves an agent uses —
# the frontmatter it routes on, and the body it follows.
meta, body = run_skill(SKILL_MD)

In [ ]:
# skills/runtime/sandbox-containment-probe/scripts/sandbox_containment_probe.py — embedded verbatim from the repository.
# This is the skill's own script, not a paraphrase of it.
#!/usr/bin/env python3
"""Execute the same code against three environments and evaluate the network policies that separate them.

This is the executable half of the `sandbox-containment-probe` skill: the check the
SKILL.md next to it describes, run against a synthetic CyberTravels
estate so two runs can be diffed and the result argued with.

Standard library only, and deterministic, so it runs on a Kaggle
kernel with the internet switched off.
"""

HOST = {"fs": ["/home/agent/work/data.csv", "/home/agent/.ssh/id_ed25519",
                "/etc/passwd"],
        "env": {"AWS_ACCESS_KEY_ID": "AKIA-EXAMPLE-NOT-REAL",
                "DATABASE_URL": "postgres://bookings-db.prod/main"},
        "net": ["bookings-db.prod:5432", "169.254.169.254:80", "0.0.0.0/0"]}

def sandbox(workdir="/sandbox/work", allow_net=(), keep_env=()):
    return {"fs": [p for p in HOST["fs"] if p.startswith(workdir)]
                  + [f"{workdir}/data.csv"],
            "env": {k: v for k, v in HOST["env"].items() if k in keep_env},
            "net": list(allow_net)}

def reach(env, code):
    out = []
    if "open("   in code: out += [f"file:{p}" for p in sorted(env["fs"])]
    if "environ" in code: out += [f"env:{k}"  for k in sorted(env["env"])]
    if "connect" in code: out += [f"net:{h}"  for h in sorted(env["net"])]
    return out

CODE = "import os; d=os.environ; open('/home/agent/.ssh/id_ed25519'); connect('x')"

configs = {
 "no sandbox":                    HOST,
 "sandbox, prod creds mounted":   sandbox(allow_net=["bookings-db.prod:5432"],
                                          keep_env=("AWS_ACCESS_KEY_ID",
                                                    "DATABASE_URL")),
 "sandbox, no ambient creds":     sandbox(),
}
for label, env in configs.items():
    r = reach(env, CODE)
    print(f"{label:32s}reached {len(r)}")
    for item in r:
        print(f"      {item}")
    print()

print("The middle configuration is the one that ships. The filesystem is")
print("isolated, the syscalls are filtered, and the environment holds a")
print("credential that reaches production over a network hop the sandbox allows.")
print()
print("Isolation of the wrong dimension is not a weaker control. It is the")
print("appearance of one.")
assert reach(configs["sandbox, no ambient creds"], CODE) and \
       not any("env:" in r for r in reach(configs["sandbox, no ambient creds"], CODE))

# The two NetworkPolicy objects above, as data. The point of modelling them
# rather than trusting them is the additive rule: a pod that NO policy selects
# is unrestricted, and policies never deny.
POLICIES = [
 {"name": "default-deny-all",      "selects": "*",              "egress": []},
 {"name": "workflow-agent-egress", "selects": "workflow-agent",
  "egress": [("bookings-db.prod", 5432), ("kube-dns", 53)]},
]

def permitted(policies, pod, dest, port):
    """Kubernetes semantics, exactly: restricted only if some policy selects
    this pod, and then permitted only if some selecting policy allows it."""
    selecting = [p for p in policies if p["selects"] in ("*", pod)]
    if not selecting:
        return True, "no policy selects this pod - unrestricted"
    for p in selecting:
        if (dest, port) in p["egress"]:
            return True, f"allowed by {p['name']}"
    return False, "no selecting policy permits it - denied"

ATTEMPTS = [
 ("workflow-agent", "bookings-db.prod", 5432, "the one it needs"),
 ("workflow-agent", "kube-dns",           53, "resolution, explicitly granted"),
 ("workflow-agent", "169.254.169.254",    80, "cloud metadata - node credentials"),
 ("workflow-agent", "archive.evil.example", 443, "A1.3's exfiltration target"),
 ("coding-agent",   "archive.evil.example", 443, "a pod nobody wrote a policy for"),
]

def run(policies, label):
    print(f"{label}:")
    out = 0
    for pod, dest, port, note in ATTEMPTS:
        ok, why = permitted(policies, pod, dest, port)
        out += ok
        print(f"   {pod:16s}{dest:22s}{port:<6d}"
              f"{'ALLOW' if ok else 'deny ':6s}{why}")
    print(f"   -> {out}/{len(ATTEMPTS)} connections permitted\n")
    return out

with_deny = run(POLICIES, "both objects applied")

# The usual breakage. Somebody removes default-deny-all because it broke a
# health check, and re-adds a targeted allow instead. Every test still passes.
without = run([p for p in POLICIES if p["name"] != "default-deny-all"],
              "default-deny-all deleted, the narrow policy kept")

print("Deleting the deny changed nothing about workflow-agent - its own policy")
print("still selects it. What it changed is every OTHER pod in the namespace,")
print("including coding-agent, which now reaches the internet because no")
print("object mentions it. Nothing failed, nothing alerted, and the namespace")
print("went from default-deny to default-allow in one commit.")
assert with_deny == 2
assert without == 3 and not permitted(POLICIES, "coding-agent",
                                      "archive.evil.example", 443)[0]

## What you just proved

The same code is executed against three environments: unsandboxed it reaches a private key, two credentials and the whole network; sandboxed with production credentials mounted it still reaches both credentials and the production database; only the third contains it. Then the two NetworkPolicy objects are evaluated — 2 of 5 connections permitted, with cloud metadata and the exfiltration target both refused for not being named. Deleting `default-deny-all` leaves workflow-agent unchanged and silently opens every other pod in the namespace.

## Your turn

Run `kubectl get networkpolicy -A` and look for a policy with an empty `podSelector` and `policyTypes: [Ingress, Egress]`. If there isn't one in the namespace your agents run in, every narrow policy you have written is an allowance rather than a restriction, and every pod nobody wrote a policy for has the internet.

---

**Next → [A3.3 · Egress control](https://spbreed.github.io/cyber-commons/lessons/A3.3.html)**

[All lessons](https://spbreed.github.io/cyber-commons/lessons/) · [This lesson's page](https://spbreed.github.io/cyber-commons/lessons/A3.2.html) · [Source](https://github.com/spbreed/cyber-commons/blob/claude/vulnbench-setup-scheduling-81aqov/labs/notebooks/A3.2.ipynb)

*Cyber Commons — a free, open commons for Cyber AI.*